In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import torch
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    logging
)
from peft import LoraConfig, get_peft_model, TaskType
from utils.evaluation import *
from utils.preprocessing import prepare_dataset, tokenize_function

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

logging.set_verbosity_error()

In [4]:
df_expert_45 = pd.read_csv("lemmatized/expert_core_45_lemmatized.csv")
df_expert_90 = pd.read_csv("lemmatized/expert_core_90_lemmatized.csv")
df_expert_180 = pd.read_csv("lemmatized/expert_core_180_lemmatized.csv")
df_test = pd.read_csv("lemmatized/test_lemmatized.csv")

dataset_expert_45 = prepare_dataset(df_expert_45)
dataset_expert_90 = prepare_dataset(df_expert_90)
dataset_expert_180 = prepare_dataset(df_expert_180)
dataset_test = prepare_dataset(df_test)

map_func = lambda examples: tokenize_function(examples, tokenizer)
tokenized_expert_45 = dataset_expert_45.map(map_func, remove_columns=['text'], batched=True)
tokenized_expert_90 = dataset_expert_90.map(map_func, remove_columns=['text'], batched=True)
tokenized_expert_180 = dataset_expert_180.map(map_func, remove_columns=['text'], batched=True)
tokenized_test = dataset_test.map(map_func, remove_columns=['text'], batched=True)

Map: 100%|██████████| 15000/15000 [00:00<00:00, 20270.24 examples/s]


In [5]:
def train_lora_model(train_ds, name, epochs, batch_size, lr, lora_r=8, lora_alpha=16):
    print(f"\n{'='*50}")
    print(f"Training LoRA: {name}")
    print(f"{'='*50}")
    
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, 
        num_labels=3, 
        problem_type="single_label_classification", 
        ignore_mismatched_sizes=True
    ).to(device)
    
    peft_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        inference_mode=False,
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=0.1,
        target_modules=["query", "value"]
    )
    
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()
    
    args = TrainingArguments(
        logging_steps=50,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=lr,
        weight_decay=0.01,
        report_to="none",
        fp16=True if device == "cuda" else False,
        seed=42,
        disable_tqdm=False,
        save_strategy="no"
    )
    
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        compute_metrics=compute_metrics
    )
    
    trainer.train()
    return trainer

In [6]:
trainer_lora_45 = train_lora_model(
    tokenized_expert_45, "45", 
    epochs=50, batch_size=5, lr=5e-3, lora_r=16, lora_alpha=16
)


Training LoRA: 45


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 1403.45it/s, Materializing param=bert.pooler.dense.weight]                              


trainable params: 60,843 || all params: 29,255,550 || trainable%: 0.2080


Step,Training Loss
50,0.421958
100,0.003916
150,0.000940
200,0.000383
250,0.000203
300,0.000181
350,0.000151
400,0.000144
450,0.000144


In [7]:
trainer_lora_90 = train_lora_model(
    tokenized_expert_90, "90", 
    epochs=50, batch_size=10, lr=1e-3, lora_r=16, lora_alpha=16
)


Training LoRA: 90


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 1466.99it/s, Materializing param=bert.pooler.dense.weight]                              


trainable params: 60,843 || all params: 29,255,550 || trainable%: 0.2080


Step,Training Loss
50,0.834467
100,0.202396
150,0.023455
200,0.007195
250,0.004520
300,0.003469
350,0.002745
400,0.002688
450,0.002193


In [8]:
trainer_lora_180 = train_lora_model(
    tokenized_expert_180, "180", 
    epochs=38, batch_size=15, lr=7e-4, lora_r=16, lora_alpha=16
)


Training LoRA: 180


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 1418.11it/s, Materializing param=bert.pooler.dense.weight]                              


trainable params: 60,843 || all params: 29,255,550 || trainable%: 0.2080


Step,Training Loss
50,0.947009
100,0.443703
150,0.163913
200,0.067917
250,0.030272
300,0.017522
350,0.015517
400,0.011814
450,0.010280


In [9]:
res_lora_45 = evaluate_trainer(trainer_lora_45, tokenized_test, "LoRA (45 examples)")


Результаты модели: LoRA (45 examples)
   Accuracy:     0.5432
   F1 Macro:     0.5484


In [10]:
res_lora_90 = evaluate_trainer(trainer_lora_90, tokenized_test, "LoRA (90 examples)")


Результаты модели: LoRA (90 examples)
   Accuracy:     0.5433
   F1 Macro:     0.5463


In [11]:
res_lora_180 = evaluate_trainer(trainer_lora_180, tokenized_test, "LoRA (180 examples)")


Результаты модели: LoRA (180 examples)
   Accuracy:     0.5794
   F1 Macro:     0.5823
